Модель для NMF

In [ ]:
import numpy as np
import librosa
import pickle
import os
from sklearn.decomposition import NMF

TARGET_SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_COMPONENTS_SPEECH = 80
N_COMPONENTS_MUSIC = 160
N_ITER = 1500

PROJECT_PATH = r'\myproject'
speech_path = os.path.join(PROJECT_PATH, 'data', 'clean_speech', 'speech_merged.wav')
music_path = os.path.join(PROJECT_PATH, 'data', 'clean_music', 'music_merged.wav')
model_path = os.path.join(PROJECT_PATH, 'models', 'nmf_model.pkl')

def get_spectrogram(audio, n_fft=N_FFT, hop_length=HOP_LENGTH):
    D = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    return np.abs(D) ** 2

speech, sr = librosa.load(speech_path, sr=TARGET_SR, mono=True)
music, _ = librosa.load(music_path, sr=TARGET_SR, mono=True)

V_speech = get_spectrogram(speech)
V_music = get_spectrogram(music)

print(f"Речь: {V_speech.shape}, Музыка: {V_music.shape}")

nmf_speech = NMF(n_components=N_COMPONENTS_SPEECH, init='random', random_state=42, max_iter=N_ITER)
nmf_speech.fit(V_speech.T)
W_speech = nmf_speech.components_.T

nmf_music = NMF(n_components=N_COMPONENTS_MUSIC, init='random', random_state=42, max_iter=N_ITER)
nmf_music.fit(V_music.T)
W_music = nmf_music.components_.T

print(f"W_speech: {W_speech.shape}, W_music: {W_music.shape}")

model_data = {
    'W_speech': W_speech,
    'W_music': W_music,
    'n_speech': N_COMPONENTS_SPEECH,
    'n_music': N_COMPONENTS_MUSIC,
    'target_sr': TARGET_SR,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'n_freq_bins': W_speech.shape[0]
}

os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Модель сохранена: {model_path}")

Алгоритм:
1. Загружаем обученную модель (W_speech и W_music) - спектральные шаблоны речи и музыки
2. Раскладываем зашумленный сигнал на речевую и музыкальную части
3. Строим частотно-зависимую маску, которая подавляет музыку
4. Применяем бинарную маску и сохраняем результат

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import pickle
import os
from scipy.ndimage import gaussian_filter1d

TARGET_SR = 16000          # Частота дискретизации
N_FFT = 2048               # Размер окна STFT
HOP_LENGTH = 512           # Шаг окна (перекрытие 75%)

PROJECT_PATH = r'\myproject'
model_path = os.path.join(PROJECT_PATH, 'models', 'nmf_model.pkl')
input_path = os.path.join(PROJECT_PATH, 'input', 'noisy_speech.wav')
output_path = os.path.join(PROJECT_PATH, 'output', 'cleaned_speech.wav')

def get_spectrogram(audio, n_fft=N_FFT, hop_length=HOP_LENGTH):

    D = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    power = np.abs(D) ** 2
    phase = np.angle(D)
    return power, phase   #power - спектр мощности (амплитуда^2) phase - фаза (нужна для обратного преобразования)

def reconstruct_audio(power, phase, hop_length=HOP_LENGTH):
    
    S = np.sqrt(power) * np.exp(1j * phase)         # Комплексный спектр
    return librosa.istft(S, hop_length=hop_length)  # Обратное STFT

with open(model_path, 'rb') as f:
    model = pickle.load(f)

W_speech = model['W_speech']         # Базисы речи: форма (частота, компоненты)
W_music = model['W_music']           # Базисы музыки: форма (частота, компоненты)
n_speech = model['n_speech']         # Количество речевых компонент
n_freq_bins = model['n_freq_bins']   # Количество частотных бинов
n_fft = model['n_fft']
hop_length = model['hop_length']

noisy, sr = librosa.load(input_path, sr=TARGET_SR, mono=True)

V_noisy, phase = get_spectrogram(noisy, n_fft=n_fft, hop_length=hop_length)

if V_noisy.shape[0] != n_freq_bins:
    V_noisy = V_noisy[:n_freq_bins, :]
    phase = phase[:n_freq_bins, :]

W = np.hstack([W_speech, W_music])

# Ищем активации H, такие что V_noisy ≈ W @ H
# Используем мультипликативные правила обновления (Lee & Seung algorithm)
H = np.random.rand(W.shape[1], V_noisy.shape[1]) + 0.1  # Случайная инициализация
for i in range(300):                                    # 300 итераций - достаточно для сходимости
    numerator = W.T @ V_noisy
    denominator = (W.T @ W) @ H + 1e-10
    H = H * (numerator / denominator)                   # Обновление по правилу


H_speech = H[:n_speech, :]      
H_music = H[n_speech:, :]       

# Реконструируем речь и музыку по отдельности
V_speech_recon = W_speech @ H_speech   
V_music_recon = W_music @ H_music      

# ПОСТРОЕНИЕ МАСКИ

BETA = 40.0          # Усиление музыкальных базисов (чем выше, тем сильнее подавление)
THRESHOLD = 0.5      # Порог бинаризации (0.5 - середина)

n_freq = V_speech_recon.shape[0]
alpha_base = np.ones(n_freq)

# Частоты в Гц
freqs = np.linspace(0, TARGET_SR / 2, n_freq)

# Настройка частотно-зависимого подавления
alpha_base[freqs < 200] = 0.05        # Низкие частоты (бас) - давим сильнее
mask_mid = (freqs >= 200) & (freqs < 3000)
alpha_base[mask_mid] = 0.15           # Средние частоты (голос) - давим слабее
alpha_base[freqs >= 3000] = 0.08      # Высокие частоты - среднее подавление

# Сглаживание по частотам (плавный переход между зонами)
alpha_smooth = gaussian_filter1d(alpha_base, sigma=10)
alpha_per_freq = alpha_smooth[:, np.newaxis]

# Мягкая маска: где речь доминирует - маска близка к 1, где музыка - близка к 0
mask_soft = V_speech_recon / (V_speech_recon + alpha_per_freq * BETA * V_music_recon + 1e-10)

# Бинарная маска:
# Значения выше порога оставляем, ниже - обнуляем
mask_binary = (mask_soft > THRESHOLD).astype(float)

# Вывод статистики маски
print(f"BETA={BETA}, THRESHOLD={THRESHOLD}")
print(f"Маска мягкая: среднее={np.mean(mask_soft):.3f}, мин={np.min(mask_soft):.3f}, макс={np.max(mask_soft):.3f}")
print(f"Маска бинарная: среднее={np.mean(mask_binary):.3f}, доля единиц={np.mean(mask_binary)*100:.1f}%")

# ПРИМЕНЕНИЕ МАСКИ 
V_clean = mask_binary * V_noisy

# ВОССТАНОВЛЕНИЕ АУДИО
cleaned_audio = reconstruct_audio(V_clean, phase, hop_length=hop_length)

# Нормализация громкости
max_val = np.max(np.abs(cleaned_audio))
if max_val > 0:
    cleaned_audio = cleaned_audio / max_val

sf.write(output_path, cleaned_audio, TARGET_SR)

print(f"Готово: {output_path}")